# VisionBridge — trained model check (Colab)

Inference/diagnosis only. This notebook does not train the model or modify VisionBridge source code.

It downloads real ISL-CSLTR sentence-level videos, extracts the same 132-d pose + 1404-d face features used by VisionBridge, runs the trained checkpoint, and diagnoses CTC blank collapse before treating the checkpoint as usable.

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path
REPO_ROOT=Path('/content/VisionBridge')
if not (REPO_ROOT/'README.md').exists(): subprocess.run(['git','clone','https://github.com/BharathWaj-K-R/VisionBridge.git',str(REPO_ROOT)],check=True)
else: subprocess.run(['git','-C',str(REPO_ROOT),'pull','--ff-only'],check=True)
BACKEND_ROOT=REPO_ROOT/'backend'
if str(BACKEND_ROOT) not in sys.path: sys.path.insert(0,str(BACKEND_ROOT))
os.chdir(REPO_ROOT)
import app
print('Repo:',REPO_ROOT)
print('Python:',sys.version.split()[0])
print('APP IMPORT: PASS')

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
MP_ENV=Path('/content/visionbridge_mp312'); MP_PYTHON=MP_ENV/'bin/python'
MPL_CONFIG=Path('/content/visionbridge_mplconfig'); MPL_CONFIG.mkdir(parents=True,exist_ok=True)
uv=shutil.which('uv')
if uv is None: subprocess.run([sys.executable,'-m','pip','install','-q','--no-cache-dir','uv'],check=True); uv=shutil.which('uv')
if uv is None: raise RuntimeError('uv is not available on PATH; restart Colab and rerun.')
if subprocess.run([uv,'python','find','3.12'],capture_output=True).returncode!=0: subprocess.run([uv,'python','install','3.12'],check=True)
if not MP_PYTHON.exists(): subprocess.run([uv,'venv','--python','3.12',str(MP_ENV)],check=True)
env=os.environ.copy(); env['MPLBACKEND']='Agg'; env['MPLCONFIGDIR']=str(MPL_CONFIG)
probe=subprocess.run([str(MP_PYTHON),'-c','import mediapipe; from mediapipe.python.solutions import holistic; print(mediapipe.__version__)'],text=True,capture_output=True,env=env)
if probe.returncode!=0 or probe.stdout.strip()!='0.10.21': subprocess.run([uv,'pip','install','--python',str(MP_PYTHON),'mediapipe==0.10.21','numpy==1.26.4','opencv-python-headless','pandas','matplotlib'],check=True,env=env)
probe=subprocess.run([str(MP_PYTHON),'-c','import sys,os,mediapipe; from mediapipe.python.solutions import holistic; print(sys.version.split()[0]); print(mediapipe.__version__); print(os.environ.get("MPLBACKEND"))'],text=True,capture_output=True,env=env)
print(probe.stdout)
if probe.returncode!=0: print(probe.stderr); raise RuntimeError('Isolated MediaPipe environment failed validation.')


In [ ]:
import torch, shutil
from app.training.isltranslate import SimpleCharTokenizer
from app.models.base_model import load_frozen_base_model, POSE_INPUT_DIM, FACE_INPUT_DIM, MAX_SEQUENCE_LENGTH
WEIGHTS=REPO_ROOT/'backend/app/models/weights/base_model.pt'; VOCAB=REPO_ROOT/'backend/app/models/weights/base_model.vocab.json'
if not WEIGHTS.exists() or not VOCAB.exists():
    from google.colab import files
    uploaded=files.upload()
    for name in ('base_model.pt','base_model.vocab.json'):
        if name not in uploaded: raise FileNotFoundError(name)
        WEIGHTS.parent.mkdir(parents=True,exist_ok=True); shutil.copy(name,WEIGHTS.parent/name)
tokenizer=SimpleCharTokenizer.load(VOCAB)
state=torch.load(WEIGHTS,map_location='cpu')
assert isinstance(state,dict) and 'output_head.weight' in state
assert int(state['output_head.weight'].shape[0])==tokenizer.vocab_size
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model=load_frozen_base_model(str(WEIGHTS),vocab_size=tokenizer.vocab_size).to(device).eval()
assert sum(p.numel() for p in model.parameters() if p.requires_grad)==0
print('Device:',device,'Vocab:',tokenizer.vocab_size)
print('CHECKPOINT VALIDATION: PASS')

In [ ]:
import glob, os, kagglehub
dataset_path=kagglehub.dataset_download('drblack00/isl-csltr-indian-sign-language-dataset')
roots=[p for p in glob.glob(os.path.join(dataset_path,'**','*Sentence_Level*'),recursive=True) if os.path.isdir(p) and 'Video' in os.path.basename(p)]
if len(roots)!=1: raise RuntimeError(f'Expected one sentence-level video directory, found {len(roots)}: {roots}')
VIDEO_ROOT=roots[0]; video_files=[]
for ext in ('*.mp4','*.MP4','*.avi','*.AVI','*.mov','*.MOV'): video_files.extend(glob.glob(os.path.join(VIDEO_ROOT,'**',ext),recursive=True))
video_files=sorted(video_files); assert video_files
TEST_VIDEOS=video_files[:10]
print('Dataset:',dataset_path); print('Total videos:',len(video_files)); print('Diagnostic videos:',len(TEST_VIDEOS))

In [ ]:
import subprocess, textwrap, numpy as np, torch, json
from pathlib import Path
from app.services.inference_service import decode_logits
from app.training.isltranslate import _downsample_to_max_length
CHECK_DIR=REPO_ROOT/'data/model_check'; CHECK_DIR.mkdir(parents=True,exist_ok=True)
helper=CHECK_DIR/'_extract_one.py'
helper.write_text(textwrap.dedent('''
import sys
from pathlib import Path
repo=Path(sys.argv[1]); video=sys.argv[2]; pose_out=Path(sys.argv[3]); face_out=Path(sys.argv[4])
sys.path.insert(0,str(repo/'backend'))
import numpy as np
from mediapipe.python.solutions import holistic
from scripts.extract_keypoints import extract_clip_keypoints
with holistic.Holistic(static_image_mode=False,model_complexity=1) as solution:
    pose,face=extract_clip_keypoints(video,solution)
assert pose.ndim==2 and pose.shape[1]==132
assert face.ndim==2 and face.shape[1]==1404
assert pose.shape[0]==face.shape[0] and pose.shape[0]>0
np.save(pose_out,pose); np.save(face_out,face)
print(pose.shape,face.shape)
'''),encoding='utf-8')
rows=[]
for idx,video in enumerate(TEST_VIDEOS):
    pose_path=CHECK_DIR/f'pose_{idx}.npy'; face_path=CHECK_DIR/f'face_{idx}.npy'
    result=subprocess.run([str(MP_PYTHON),str(helper),str(REPO_ROOT),video,str(pose_path),str(face_path)],text=True,capture_output=True,env=env)
    if result.returncode!=0: print('EXTRACTION FAILED',video,result.stderr); continue
    pose_np=np.load(pose_path); face_np=np.load(face_path)
    pose_t,face_t=_downsample_to_max_length(torch.from_numpy(pose_np).float(),torch.from_numpy(face_np).float(),Path(video).stem)
    with torch.inference_mode(): logits=model(pose_t.unsqueeze(0).to(device),face_t.unsqueeze(0).to(device))
    probs=torch.softmax(logits,dim=-1); ids=probs.argmax(dim=-1)[0]
    blank_ratio=float((ids==0).float().mean())
    nonblank=[int(x) for x in ids.tolist() if int(x)!=0]
    pred,conf=decode_logits(logits)
    truth=Path(video).parent.name.replace('_',' ').strip()
    rows.append({'video':Path(video).name,'truth':truth,'prediction':pred,'confidence':float(conf),'blank_ratio':blank_ratio,'nonblank_count':len(nonblank),'unique_nonblank':len(set(nonblank)),'logits_finite':bool(torch.isfinite(logits).all())})
    print(f"[{idx+1}/{len(TEST_VIDEOS)}] blank_ratio={blank_ratio:.3f} nonblank={len(nonblank)} pred={pred!r}")
assert rows, 'No real videos were successfully processed.'
mean_blank=sum(r['blank_ratio'] for r in rows)/len(rows)
invalid=sum(1 for r in rows if not r['logits_finite'])
empty=sum(1 for r in rows if r['prediction']=='(no sign detected)')
print('\nDIAGNOSTIC SUMMARY')
print('Samples:',len(rows))
print('Mean blank ratio:',round(mean_blank,4))
print('Empty predictions:',f'{empty}/{len(rows)}')
print('Invalid logits:',f'{invalid}/{len(rows)}')
if invalid: print('CASE D: numerical/logit problem')
elif mean_blank>=0.99: print('CASE A: checkpoint is collapsed to CTC blank on these real samples — retraining is required after fixing training/data causes.')
elif mean_blank>=0.75: print('CASE B: high blank ratio — inspect preprocessing/sequence consistency and training health before retraining.')
else: print('CASE B/C: model is producing non-blank tokens; inspect decoder/tokenizer and per-sample preprocessing.')
print(json.dumps(rows,indent=2))


In [ ]:
r=rows[0]
print('GROUND TRUTH:',r['truth'])
print('PREDICTED:   ',r['prediction'])
print('CONFIDENCE:  ',round(r['confidence'],4))
print('BLANK RATIO: ',round(r['blank_ratio'],4))
print('NON-BLANK:   ',r['nonblank_count'])
print('UNIQUE TOKENS:',r['unique_nonblank'])
print('LOGITS FINITE:',r['logits_finite'])